# Brain-Tumor Classifiers - Full Comparison

Loads **all 8 trained models** (4 architectures x {pretrained, scratch}) and evaluates
them on the shared Roboflow test set, so the team can compare every model in one place.

| architecture | pretrained ckpt | scratch ckpt | input | loss / decision |
|---|---|---|---|---|
| SqueezeNet   | `squeezenet.pt`          | `squeezenet_s.pt`        | 227px | CrossEntropy / argmax |
| ResNet-18    | `resnet18.pt`            | `resnet_custom.pt`       | 227px | CrossEntropy / argmax |
| EfficientNet | `efficientnet.pt`        | `efficientnet_scratch.pt`| 224px | CrossEntropy / argmax |
| GoogLeNet    | `googlenet_pretrained.pth`| `googlenet_scratch.pth` | 224px | BCEWithLogits / sigmoid>=0.5 |

**The model architectures below are copied verbatim from each teammate's branch** -
nothing about their characteristics is changed. Each architecture is rebuilt exactly
as it was trained, then its saved `state_dict` is loaded.

> Note: `resnet_custom.pt` (ResNet from scratch) has not been trained yet, so that one
> row is skipped automatically until the checkpoint exists.

## 0. Setup & configuration

Auto-detects the data + checkpoint locations so the notebook runs on Kaggle (datasets
mounted under `/kaggle/input`) or locally (`brain-Tumor-1/` + `comparison_checkpoints/`).

In [ ]:
import os, glob, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from torchvision.models import (
    squeezenet1_0, SqueezeNet1_0_Weights,
    resnet18, ResNet18_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    googlenet, GoogLeNet_Weights,
)
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

def _glob_first(name, kind="file"):
    """Recursively find a file/dir by name under the usual Kaggle/local roots.

    Works no matter how a dataset is mounted (/kaggle/input/<name>,
    /kaggle/input/datasets/<owner>/<name>, a local folder, ...).
    """
    for base in ("/kaggle/input", "/kaggle/working", "."):
        if not os.path.isdir(base):
            continue
        for hit in sorted(glob.glob(os.path.join(base, "**", name), recursive=True)):
            if (kind == "file" and os.path.isfile(hit)) or (kind == "dir" and os.path.isdir(hit)):
                return hit
    return None

# Long's preprocess.py supplies ImagePreprocessor (the shared train/test transforms).
pp = _glob_first("preprocess.py", "file")
assert pp, "preprocess.py not found - attach the 'brain-tumor-detection-support' dataset"
sys.path.insert(0, os.path.dirname(pp))
from preprocess import ImagePreprocessor

# Test images: <DATA_ROOT>/test/<class>/*.png  (ImageFolder layout).
test_dir = _glob_first("test", "dir")
assert test_dir, "test/ folder not found - attach the 'brain-tumor-roboflow-v1' dataset"
DATA_ROOT = os.path.dirname(test_dir)

# Checkpoints (.pt/.pth) are resolved by filename, recursively, wherever attached.
# Returns None if a checkpoint is missing -> that model is skipped (no error).
def find_ckpt(fname):
    for base in ("comparison_checkpoints", "/kaggle/input", "/kaggle/working", "."):
        if not os.path.isdir(base):
            continue
        direct = os.path.join(base, fname)
        if os.path.isfile(direct):
            return direct
        hits = sorted(glob.glob(os.path.join(base, "**", fname), recursive=True))
        if hits:
            return hits[0]
    return None

print("preprocess.py:", pp)
print("DATA_ROOT    :", DATA_ROOT)

## 1. Model definitions (verbatim)

The two hand-written architectures are imported from their own modules rather than
pasted in, so this notebook stays lean: GoogLeNet from `googlenet.py` (GoogLeNet branch)
and the from-scratch `SmallResNet` from `small_resnet.py` (ResNet branch). The other six
models are plain torchvision builders, defined in the registry below.

In [ ]:
# GoogLeNet (Inception v1) lives verbatim in googlenet.py (GoogLeNet branch, by the
# GoogLeNet owner). Instead of pasting the whole module in, we locate the file wherever
# it is attached and import it - keeps this notebook lean.
gnet_py = _glob_first("googlenet.py", "file")
assert gnet_py, "googlenet.py not found - attach the dataset that ships it (brain-tumor-checkpoints)"
sys.path.insert(0, os.path.dirname(gnet_py))
from googlenet import GoogLeNet, InceptionModule, AuxiliaryClassifier
print("googlenet.py:", gnet_py)

In [ ]:
# SmallResNet (hand-written from-scratch ResNet, ResNet branch) lives verbatim in
# small_resnet.py. Instead of pasting it in, we locate the file wherever it is attached
# and import it - keeps this notebook lean.
sr_py = _glob_first("small_resnet.py", "file")
assert sr_py, "small_resnet.py not found - attach the dataset that ships it (brain-tumor-checkpoints)"
sys.path.insert(0, os.path.dirname(sr_py))
from small_resnet import SmallResNet, BasicBlock
print("small_resnet.py:", sr_py)

## 2. Shared test set

One test set, loaded at each model's native input size (227px or 224px). Labels come
from `ImageFolder` (alphabetical class order), identical for every model, so accuracy
is directly comparable.

In [ ]:
SIZES = [(227, 227), (224, 224)]
test_loaders = {}
for size in SIZES:
    tf = ImagePreprocessor(size=size).test_transform
    ds = ImageFolder(os.path.join(DATA_ROOT, "test"), transform=tf)
    test_loaders[size] = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=False)

CLASSES = test_loaders[SIZES[0]].dataset.classes
NUM_CLASSES = len(CLASSES)
POS_LABEL = NUM_CLASSES - 1            # second class (alphabetical) = positive (Tumor)
print("classes:", CLASSES, "| positive label:", CLASSES[POS_LABEL],
      "| test images:", len(test_loaders[SIZES[0]].dataset))

## 3. Model registry & builders

Each builder reconstructs an architecture **exactly** as its training notebook did
(same constructor, same replaced head). Pretrained constructors keep their original
`weights=...DEFAULT` argument; those weights are immediately overwritten by the loaded
checkpoint (Kaggle has internet enabled, so the one-time download is harmless).

In [ ]:
def build_squeezenet_pretrained():
    m = squeezenet1_0(weights=SqueezeNet1_0_Weights.DEFAULT)
    m.classifier[1] = torch.nn.Conv2d(512, NUM_CLASSES, kernel_size=(1, 1), stride=(1, 1))
    return m

def build_squeezenet_scratch():
    m = squeezenet1_0()
    m.classifier[1] = torch.nn.Conv2d(512, NUM_CLASSES, kernel_size=(1, 1), stride=(1, 1))
    return m

def build_resnet_pretrained():
    m = resnet18(weights=ResNet18_Weights.DEFAULT)
    m.fc = torch.nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

def build_resnet_scratch():
    return SmallResNet(NUM_CLASSES)

def build_efficientnet_pretrained():
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    m.classifier[1] = torch.nn.Linear(in_features=1280, out_features=NUM_CLASSES)
    return m

def build_efficientnet_scratch():
    m = efficientnet_b0(weights=None)
    m.classifier[1] = torch.nn.Linear(in_features=1280, out_features=NUM_CLASSES)
    return m

def build_googlenet_pretrained():
    m = googlenet(weights=GoogLeNet_Weights.IMAGENET1K_V1, aux_logits=True)
    m.fc = nn.Linear(m.fc.in_features, 1)
    m.aux1.fc2 = nn.Linear(m.aux1.fc2.in_features, 1)
    m.aux2.fc2 = nn.Linear(m.aux2.fc2.in_features, 1)
    return m

def build_googlenet_scratch():
    return GoogLeNet(lr=1e-4, weight_decay=1e-4, dropout_prob=0.4, aux_weight=0.3)

REGISTRY = [
    {"name": "SqueezeNet",   "variant": "pretrained", "build": build_squeezenet_pretrained,   "ckpt": "squeezenet.pt",             "size": (227, 227), "head": "ce"},
    {"name": "SqueezeNet",   "variant": "scratch",    "build": build_squeezenet_scratch,      "ckpt": "squeezenet_s.pt",           "size": (227, 227), "head": "ce"},
    {"name": "ResNet-18",    "variant": "pretrained", "build": build_resnet_pretrained,       "ckpt": "resnet18.pt",               "size": (227, 227), "head": "ce"},
    {"name": "ResNet",       "variant": "scratch",    "build": build_resnet_scratch,          "ckpt": "resnet_custom.pt",          "size": (227, 227), "head": "ce"},
    {"name": "EfficientNet", "variant": "pretrained", "build": build_efficientnet_pretrained, "ckpt": "efficientnet.pt",           "size": (224, 224), "head": "ce"},
    {"name": "EfficientNet", "variant": "scratch",    "build": build_efficientnet_scratch,    "ckpt": "efficientnet_scratch.pt",   "size": (224, 224), "head": "ce"},
    {"name": "GoogLeNet",    "variant": "pretrained", "build": build_googlenet_pretrained,    "ckpt": "googlenet_pretrained.pth",  "size": (224, 224), "head": "bce"},
    {"name": "GoogLeNet",    "variant": "scratch",    "build": build_googlenet_scratch,       "ckpt": "googlenet_scratch.pth",     "size": (224, 224), "head": "bce"},
]
print(len(REGISTRY), "models registered")

## 4. Load checkpoints & evaluate

For each model: rebuild -> load `state_dict` -> predict on the test set. CrossEntropy
models decide with `argmax`; GoogLeNet (single logit, BCE) decides with `sigmoid>=0.5`.
Missing checkpoints are skipped with a message instead of erroring.

In [ ]:
@torch.no_grad()
def predict(model, loader, head):
    model.eval()
    y_true, y_pred = [], []
    for x, y in loader:
        x = x.to(device)
        out = model(x)
        if isinstance(out, tuple):          # GoogLeNet can return (main, aux1, aux2)
            out = out[0]
        if head == "ce":
            p = out.argmax(1)
        else:                                # bce: single logit -> sigmoid -> 0.5
            p = (torch.sigmoid(out) >= 0.5).long().squeeze(1)
        y_pred += p.cpu().tolist()
        y_true += y.tolist()
    return np.array(y_true), np.array(y_pred)

results, conf_mats = [], {}
for spec in REGISTRY:
    tag = f"{spec['name']} ({spec['variant']})"
    path = find_ckpt(spec["ckpt"])
    if path is None:
        print(f"SKIP  {tag:26s}  ->  '{spec['ckpt']}' not found")
        continue

    model = spec["build"]().to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    n_params = sum(p.numel() for p in model.parameters())

    y_true, y_pred = predict(model, test_loaders[spec["size"]], spec["head"])
    acc = float((y_true == y_pred).mean())
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=POS_LABEL, zero_division=0)
    conf_mats[tag] = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

    results.append({
        "model": spec["name"], "variant": spec["variant"],
        "params": n_params, "input": f"{spec['size'][0]}px", "loss": spec["head"].upper(),
        "accuracy": round(acc, 4),
        "precision": round(float(prec), 4),
        "recall": round(float(rec), 4),
        "f1": round(float(f1), 4),
    })
    print(f"OK    {tag:26s}  acc={acc:.4f}  params={n_params:,}")

    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

print(f"\nevaluated {len(results)} / {len(REGISTRY)} models "
      f"(positive class = '{CLASSES[POS_LABEL]}')")

## 5. Results table

Sorted by test accuracy. `precision` / `recall` / `f1` are for the positive (Tumor) class.

In [ ]:
if results:
    df = pd.DataFrame(results).sort_values("accuracy", ascending=False).reset_index(drop=True)
    show = df.copy()
    show["params"] = show["params"].map(lambda n: f"{n:,}")
    print(show.to_string(index=False))
else:
    df = pd.DataFrame()
    show = df
    print("No models were evaluated - no checkpoints were found under the search roots.")
    print("Attach the checkpoint dataset (the 8 .pt/.pth files) and re-run to fill this table.")
show

## 6. Plots

Accuracy ranking (pretrained vs scratch) and a confusion-matrix grid.

In [ ]:
if results:
    order = df.sort_values("accuracy")
    labels = [f"{r['model']} ({r['variant']})" for _, r in order.iterrows()]
    colors = ["#4c72b0" if v == "pretrained" else "#dd8452" for v in order["variant"]]

    plt.figure(figsize=(10, 5))
    bars = plt.barh(labels, order["accuracy"], color=colors)
    plt.xlim(0, 1); plt.xlabel("test accuracy")
    plt.title("Brain-tumor classifiers - test accuracy (blue=pretrained, orange=scratch)")
    for b, v in zip(bars, order["accuracy"]):
        plt.text(v + 0.01, b.get_y() + b.get_height() / 2, f"{v:.3f}", va="center")
    plt.tight_layout(); plt.show()

    n = len(conf_mats); cols = 4; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (tag, cm) in zip(axes, conf_mats.items()):
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
        ax.set_title(tag, fontsize=10); ax.set_xlabel("pred"); ax.set_ylabel("true")
    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("no results to plot - no checkpoints were found")